# Titanic Baseline EDA

この notebook は、Titanic の最初の探索とベースライン作成を 1 本で確認するためのものです。

ここでやること:
- `train.csv` / `test.csv` を読む
- 欠損と基本分布を確認する
- 最小構成のベースラインを CV で評価する
- 提出用 CSV を作る

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

SRC_DIR = COMP_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from features import TitanicFeatureEngineer
from preprocess import build_preprocessor

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "notebook_baseline_submission.csv"

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
display(train_df.head())

In [ ]:
summary_df = pd.DataFrame(
    {
        "dtype": train_df.dtypes.astype(str),
        "missing": train_df.isna().sum(),
        "missing_rate": (train_df.isna().mean() * 100).round(2),
        "n_unique": train_df.nunique(),
    }
).sort_values("missing", ascending=False)

display(summary_df)

In [ ]:
print("Survived distribution")
display(train_df["Survived"].value_counts(normalize=True).rename("ratio").to_frame())

print("Survival by Sex")
display(train_df.groupby("Sex")["Survived"].mean().sort_values(ascending=False).to_frame("survival_rate"))

print("Survival by Pclass")
display(train_df.groupby("Pclass")["Survived"].mean().sort_index().to_frame("survival_rate"))

print("Numeric summary")
display(train_df[["Age", "Fare", "SibSp", "Parch"]].describe())

## Baseline

この notebook のベースラインは、既存の `src/features.py` と `src/preprocess.py` を呼び出し、`LogisticRegression` を使う最小構成です。

最初の目的はスコアを最大化することではなく、以後の比較対象を作ることです。

In [ ]:
TARGET_COL = "Survived"
ID_COL = "PassengerId"

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype(int)

pipeline = Pipeline(
    steps=[
        ("features", TitanicFeatureEngineer()),
        ("preprocess", build_preprocessor()),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))

In [ ]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df).astype(int)

submission_df = pd.DataFrame(
    {
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())

## Next Step Ideas

- `Age`, `Cabin`, `Embarked` の欠損を詳しく見る
- `Name` の敬称や `FamilySize` が効いているかを確認する
- `LightGBM` を使う場合は、この notebook の前処理方針を基準に置き換えて比較する
- よかった構成だけ `src/train.py` / `src/predict.py` に反映する